In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

# Guardem el target abans de res
y = np.log1p(train['SalePrice'])

# Treiem SalePrice i Id per treballar còmode
train.drop(['SalePrice', 'Id'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

# Combinem train i test per fer el preprocessing igual als dos
df = pd.concat([train, test], axis=0).reset_index(drop=True)

print(f"Dataset combinat: {df.shape}")

Dataset combinat: (2919, 79)


In [2]:
# Nuls que signifiquen "absent" → omplir amb "None" o 0

# Categòriques — "None" = no té
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
             'BsmtFinType2', 'MasVnrType']

for col in cols_none:
    df[col] = df[col].fillna('None')

# Numèriques — 0 = no té
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_zero:
    df[col] = df[col].fillna(0)

print("Nuls semàntics tractats!")
print(df[cols_none + cols_zero].isnull().sum().sum(), "nuls restants en aquestes columnes")

Nuls semàntics tractats!
0 nuls restants en aquestes columnes


In [3]:
# LotFrontage — mediana per Neighborhood (recordes el brainstorm?)
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage']\
                      .transform(lambda x: x.fillna(x.median()))

df['MasVnrArea'] = df.groupby('Neighborhood')['MasVnrArea']\
                     .transform(lambda x: x.fillna(x.median()))

df['Electrical'] = df.groupby('Neighborhood')['Electrical']\
                     .transform(lambda x: x.fillna(x.mode()[0]))

print("Nuls reals tractats!")
print("Nuls restants al dataset:", df.isnull().sum().sum())

Nuls reals tractats!
Nuls restants al dataset: 12


In [4]:
df.isnull().sum()[df.isnull().sum() > 0]

MSZoning       4
Utilities      2
Exterior1st    1
Exterior2nd    1
KitchenQual    1
Functional     2
SaleType       1
dtype: int64

In [5]:
cols_moda = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 
             'KitchenQual', 'Functional', 'SaleType']

for col in cols_moda:
    df[col] = df.groupby('Neighborhood')[col]\
                .transform(lambda x: x.fillna(x.mode()[0]))

print("Nuls restants:", df.isnull().sum().sum())

Nuls restants: 0


In [6]:
# Mapeig de qualitat — el mateix ordre per totes les variables de qualitat
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

cols_qual = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
             'HeatingQC', 'KitchenQual', 'FireplaceQu',
             'GarageQual', 'GarageCond', 'PoolQC']

for col in cols_qual:
    df[col] = df[col].map(qual_map)

# Altres ordinals amb el seu propi ordre
df['BsmtExposure'] = df['BsmtExposure'].map({'None':0, 'No':1, 'Mn':2, 'Av':3, 'Gd':4})
df['BsmtFinType1'] = df['BsmtFinType1'].map({'None':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6})
df['BsmtFinType2'] = df['BsmtFinType2'].map({'None':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6})
df['GarageFinish'] = df['GarageFinish'].map({'None':0, 'Unf':1, 'RFn':2, 'Fin':3})
df['Functional']   = df['Functional'].map({'Sal':1, 'Sev':2, 'Maj2':3, 'Maj1':4, 'Mod':5, 'Min2':6, 'Min1':7, 'Typ':8})

print("Ordinals tractats!")
print(df[cols_qual].head())

Ordinals tractats!
   ExterQual  ExterCond  BsmtQual  BsmtCond  HeatingQC  KitchenQual  \
0          4          3         4         3          5            4   
1          3          3         4         3          5            3   
2          4          3         4         3          5            4   
3          3          3         3         4          4            4   
4          4          3         4         3          5            4   

   FireplaceQu  GarageQual  GarageCond  PoolQC  
0            0           3           3       0  
1            3           3           3       0  
2            3           3           3       0  
3            4           3           3       0  
4            3           3           3       0  


In [8]:
# Identifiquem les columnes categòriques restants
cat_cols = df.select_dtypes(include='str').columns.tolist()
print(f"Columnes categòriques: {len(cat_cols)}")
print(cat_cols)

Columnes categòriques: 28
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [9]:
df = pd.get_dummies(df, columns=cat_cols)

print(f"Shape després del One-Hot: {df.shape}")

Shape després del One-Hot: (2919, 237)
